# Nature Risk Screening for an Asset Portfolio, Cecil + Wherobots

**One data contract. One query engine. One assistant.**

This notebook screens a portfolio of mining assets for nature-related risk by
combining **vector** and **raster** environmental data in a *single* spatial-SQL
session. The point of the demo is not scale, it is **how little friction** there
is to do raster + vector analysis together on one platform.

The same question would normally require stitching together several tools:
`GDAL`/`rasterio` to read and reproject rasters, `geopandas`/`PostGIS` for the
vector joins, and custom glue to align coordinate systems, resolutions, and
NoData between them. Here it is a dozen SQL statements.

### The workflow (mirrors Cecil's mining-portfolio example)

1. **Locate** the assets, an internal registry of asset coordinates → buffered
   areas of influence (vector).
2. **Access** the risk layers, one Cecil integration delivers both raster and
   vector datasets for the area of interest.
3. **Analyze** in Wherobots, interleave `ST_*` (vector) and `RS_*` (raster)
   operators in one SQL session to produce a per-asset nature-risk score.

### Datasets

| Layer | Type | Provider (via Cecil) | Role |
|---|---|---|---|
| Asset registry | Vector (points → buffers) | Internal / illustrative | Areas of influence |
| Protected & Conserved Areas (WDPA) | Vector | IBAT | Proximity to protected land |
| Key Biodiversity Areas (KBA) | Vector | IBAT | Overlap with priority biodiversity |
| Biodiversity Intactness Index, 1 km | Raster | NHM | Ecological value in the footprint |
| Hansen Global Forest Change, 30 m | Raster | UMD | Recent deforestation in the footprint |

### Prerequisites
- A Wherobots Cloud notebook environment
- A Cecil API key (set as `CECIL_API_KEY`)

> **Cost note.** WDPA, KBA (IBAT) and BII (NHM) are commercial datasets on Cecil;
> Hansen is free. This demo uses a **small AOI** to keep any acquisition cost
> negligible. For a fully free variant, swap WDPA for USGS PAD-US (US only) and
> drop BII.

## The three "ease" wins to watch for

1. **One data contract (Cecil).** Four datasets from four providers arrive
   through the *same* SDK flow, `create_aoi` → `create_subscription` → scoped
   S3 credentials, instead of four bespoke contracts, formats, and download
   paths.
2. **One engine, one language (Wherobots).** `RS_FromPath` reads the Cecil
   GeoTIFFs **in place from S3** (no download, no format conversion). Then
   `RS_ZonalStats` (raster) and `ST_Distance`/`ST_Intersects` (vector) run in the
   **same SQL query**, and `RS_ZonalStats` reprojects the query geometry to each
   raster's CRS automatically, so a 1 km biodiversity grid and a 30 m forest grid
   join to the same asset buffers with **no manual reprojection or resampling**.
3. **One assistant.** Everything below can be generated from a plain-English
   prompt via the Wherobots Spatial Assistant / the `cecil-for-wherobots` skill,    see the final section.

## 1. Setup

In [ ]:
%pip install cecil

In [ ]:
import json
import os

import boto3
import cecil
import pandas as pd
from cecil.models.subscription import SubscriptionFormat, SubscriptionTIFF
from pyspark.sql import functions as F
from sedona.spark import SedonaContext

config = SedonaContext.builder().getOrCreate()
sedona = SedonaContext.create(config)

## 2. Locate the assets

Facility locations typically live in an internal asset registry as lat/lon
coordinates. Here we use an **illustrative** registry of well-known operations in
the Carajás mineral province (Pará, Brazil), a region where mining sits close to
Amazon protected areas and active deforestation, so every risk layer is
meaningful. Coordinates are approximate and for demonstration only.

We turn each point into a 10 km **area of influence** with `ST_Buffer`. We buffer
in a metric CRS (UTM 22S, EPSG:32722) and return to WGS84, the one bit of CRS
handling we do explicitly; everything downstream is automatic.

In [ ]:
# Illustrative asset registry: (id, name, lon, lat)
mine_records = [
    (1, "Serra Norte",     -50.16, -6.06),
    (2, "Serra Sul S11D",  -50.30, -6.20),
    (3, "Igarape Bahia",   -49.90, -5.95),
    (4, "Sossego",         -50.05, -6.35),
    (5, "Salobo",          -49.70, -6.10),
    (6, "Onca Puma",       -50.40, -5.80),
]

mines_pdf = pd.DataFrame(mine_records, columns=["mine_id", "name", "lon", "lat"])
mines_sdf = sedona.createDataFrame(mines_pdf)

# Point geometry (WGS84), then a 10 km buffer built in a metric CRS
mines_sdf = (
    mines_sdf
    .withColumn("geom", F.expr("ST_SetSRID(ST_Point(lon, lat), 4326)"))
    .withColumn(
        "buffer",
        F.expr(
            "ST_Transform("
            "  ST_Buffer(ST_Transform(geom, 'EPSG:4326', 'EPSG:32722'), 10000.0),"
            "  'EPSG:32722', 'EPSG:4326')"
        ),
    )
)
mines_sdf.cache()
mines_sdf.createOrReplaceTempView("mines")
print(f"{mines_sdf.count()} assets loaded")
mines_sdf.select("mine_id", "name", "lon", "lat").show(truncate=False)

### Optional: source the portfolio from Cecil's Global Major Mines

Instead of the static registry above, you can pull real asset locations from
Cecil's **Global Major Mines** dataset (verified point locations with owner,
operator, and commodity attributes). It is a **commercial** dataset ("Contact us
for access"), so the cell below is **commented out**, the static locations above
are used for testing. Uncomment it (after the Cecil client is initialized in
Step 3) to replace the portfolio with live data; it produces the same `mines`
view (`mine_id`, `name`, `geom`, `buffer`) so nothing downstream changes.

In [ ]:
# OPTIONAL, live asset locations from Cecil (COMMERCIAL dataset).
# Requires `cecil_client` from Step 3. Uncomment to use instead of the static registry.
#
# GLOBAL_MAJOR_MINES_ID = "36ae644b-11ab-4977-bbc6-78b3226b440d"  # Vector, Commercial (IBAT contact-for-access)
#
# # 1. A discovery AOI to search for mines (Polygon/MultiPolygon). Here: the Carajas region.
# discovery_geojson = {
#     "type": "Polygon",
#     "coordinates": [[
#         [-51.0, -7.0], [-49.0, -7.0], [-49.0, -5.0], [-51.0, -5.0], [-51.0, -7.0],
#     ]],
# }
# mines_aoi = cecil_client.create_aoi(external_ref="Mine discovery - Carajas", geometry=discovery_geojson)
# mines_sub = cecil_client.create_subscription(
#     external_ref="Global Major Mines - Carajas",
#     aoi_id=mines_aoi.id,
#     dataset_id=GLOBAL_MAJOR_MINES_ID,
# )
#
# # 2. Load as a Sedona DataFrame. This dataset carries lat/lon columns (point centroids),
# #    so we build geometry from those rather than a geojson column.
# mines_pdf = cecil_client.load_dataframe(mines_sub.id)
# mines_sdf = sedona.createDataFrame(mines_pdf)
#
# # 3. Optional filters (by commodity, status, ownership confidence, etc.)
# # mines_sdf = mines_sdf.filter(F.col("status") == "Operating")
#
# # 4. Conform to the schema the rest of the notebook expects
# mines_sdf = (
#     mines_sdf
#     .withColumnRenamed("mine_name", "name")
#     .withColumn("lon", F.col("longitude"))
#     .withColumn("lat", F.col("latitude"))
#     .withColumn("geom", F.expr("ST_SetSRID(ST_Point(longitude, latitude), 4326)"))
#     .withColumn(
#         "buffer",
#         F.expr(
#             "ST_Transform("
#             "  ST_Buffer(ST_Transform(geom, 'EPSG:4326', 'EPSG:32722'), 10000.0),"
#             "  'EPSG:32722', 'EPSG:4326')"
#         ),
#     )
# )
# mines_sdf.cache()
# mines_sdf.createOrReplaceTempView("mines")
# print(f"{mines_sdf.count()} mines loaded from Cecil")
# mines_sdf.select("mine_id", "name", "core_commodities", "direct_owner", "status").show(truncate=40)

In [ ]:
# Area of interest for Cecil = bounding box of the buffered assets (+ padding)
PAD = 0.15  # degrees
min_lon = min(r[2] for r in mine_records) - PAD
max_lon = max(r[2] for r in mine_records) + PAD
min_lat = min(r[3] for r in mine_records) - PAD
max_lat = max(r[3] for r in mine_records) + PAD

aoi_geojson = {
    "type": "Polygon",
    "coordinates": [[
        [min_lon, min_lat],
        [max_lon, min_lat],
        [max_lon, max_lat],
        [min_lon, max_lat],
        [min_lon, min_lat],
    ]],
}
print(f"AOI bbox: lon [{min_lon:.2f}, {max_lon:.2f}], lat [{min_lat:.2f}, {max_lat:.2f}]")

## 3. Access the risk layers, one Cecil integration

Every dataset, raster or vector, free or commercial, is acquired through the
same three calls: `create_aoi`, `create_subscription`, then fetch scoped
credentials. Two small helpers capture the raster and vector paths so the rest of
the notebook stays declarative.

In [ ]:
import getpass

# Prompts at runtime and masks input, the key is never stored in the notebook
os.environ["CECIL_API_KEY"] = getpass.getpass("Cecil API Key: ")

cecil_client = cecil.Client()

aoi = cecil_client.create_aoi(
    external_ref="Carajas mining portfolio",
    geometry=aoi_geojson,
)
print(f"AOI ID: {aoi.id}  ({aoi.hectares:,.0f} ha)")

# Cecil dataset IDs (from cecil_client.list_datasets())
DATASETS = {
    "bii":       "169344ec-bccd-4891-b6cd-48dbeabb8482",  # Biodiversity Intactness Index 1km (raster, NHM)
    "hansen":    "9659ec1d-7091-4f8b-9db5-e9fe07d2f508",  # Hansen Global Forest Change 30m (raster, UMD)
    "wdpa":      "c1ee0d62-95ef-49b1-adf1-6a5a933d726d",  # Protected & Conserved Areas (vector, IBAT)
    "kba":       "528f54b8-1cb9-412c-a646-30a55ddb7cbd",  # Key Biodiversity Areas (vector, IBAT)
}

subs = {}
for key, dataset_id in DATASETS.items():
    s = cecil_client.create_subscription(
        external_ref=f"Carajas - {key}",
        aoi_id=aoi.id,
        dataset_id=dataset_id,
    )
    subs[key] = s.id
    print(f"  {key:8s} subscription: {s.id}")

> **Subscriptions are asynchronous.** Data may take minutes to hours to become
> available. If a credential fetch below returns empty, wait and re-run that cell.
> If you already have subscription IDs, set `subs = {...}` directly and skip the
> cell above.

In [ ]:
def load_cecil_rasters(subscription_id):
    """Return a cached Sedona DataFrame of Cecil GeoTIFFs as out-of-DB rasters."""
    # Confirm GeoTIFF (RS_FromPath does not read Zarr yet)
    fmt = SubscriptionFormat(
        **cecil_client._get(url="/v0/dataset-format",
                            params={"subscription_id": subscription_id})
    )
    assert fmt.format == "tiff", (
        f"Subscription {subscription_id} is '{fmt.format}'. RS_FromPath needs GeoTIFF; "
        "use cecil_client._load_xarray_v2() for Zarr datasets."
    )

    res = SubscriptionTIFF(
        **cecil_client._get(url=f"/v0/subscriptions/{subscription_id}/files/tiff")
    )
    session = boto3.session.Session(
        aws_access_key_id=res.credentials.access_key_id,
        aws_secret_access_key=res.credentials.secret_access_key,
        aws_session_token=res.credentials.session_token,
        region_name=res.credentials.region,
    )
    s3 = session.client("s3")
    paginator = s3.get_paginator("list_objects_v2")
    keys = [
        obj["Key"]
        for page in paginator.paginate(Bucket=res.bucket.name, Prefix=res.bucket.prefix)
        for obj in page.get("Contents", [])
        if obj["Key"].lower().endswith((".tif", ".tiff"))
    ]

    cred_params = (
        f"fs.s3a.access.key={res.credentials.access_key_id};"
        f"fs.s3a.secret.key={res.credentials.secret_access_key};"
        f"fs.s3a.session.token={res.credentials.session_token}"
    )
    paths = [f"s3a://{res.bucket.name}/{k}" for k in keys]
    rdf = (
        sedona.createDataFrame([(p,) for p in paths], ["path"])
        .withColumn("rast", F.expr(f"RS_FromPath(path, '{cred_params}')"))
    )
    rdf.cache()
    print(f"  loaded {rdf.count()} GeoTIFF(s) for {res.dataset_name}")
    return rdf


def load_cecil_vector(subscription_id):
    """Return a cached Sedona DataFrame for a Cecil vector (Parquet) dataset."""
    pdf = cecil_client.load_dataframe(subscription_id)
    sdf = (
        sedona.createDataFrame(pdf)
        .withColumn("geometry", F.expr("ST_GeomFromGeoJSON(geojson)"))
        .drop("geojson")
    )
    sdf.cache()
    print(f"  loaded {sdf.count()} features")
    return sdf

### 3a. Raster layers, read in place from S3, no conversion

`RS_FromPath` registers each Cecil GeoTIFF as an out-of-database raster; pixels
are fetched on demand via HTTP range requests. Hansen is delivered as one file
per variable (`tree_cover`, `loss_year`, ...), so we register a view per band.

In [ ]:
# Biodiversity Intactness Index (single raster, band 1 = biodiversity_intactness_index)
bii_rdf = load_cecil_rasters(subs["bii"])
bii_rdf.createOrReplaceTempView("bii")

# Hansen: one GeoTIFF per variable, register a single-band view for each
hansen_rdf = load_cecil_rasters(subs["hansen"])
for variable in ["tree_cover", "loss_year", "forest_gain", "data_mask"]:
    hansen_rdf.filter(F.col("path").contains(variable)) \
        .createOrReplaceTempView(variable)

# Inspect band metadata so you can confirm band numbers
hansen_rdf.selectExpr("path", "RS_NumBands(rast) AS bands", "RS_SRID(rast) AS srid") \
    .show(truncate=80)

### 3b. Vector layers

In [ ]:
wdpa_sdf = load_cecil_vector(subs["wdpa"])
wdpa_sdf.createOrReplaceTempView("wdpa")

kba_sdf = load_cecil_vector(subs["kba"])
kba_sdf.createOrReplaceTempView("kba")

wdpa_sdf.select("name", "designation", "iucn_category", "status").show(5, truncate=40)

## 4. Analyze, raster and vector in one SQL session

Now the payoff. Each query below mixes geometry and raster operators freely, over
the same asset buffers, with no reprojection or format wrangling in between.

**Vector × vector**, proximity to protected land and key biodiversity areas.
`ST_DistanceSpheroid` returns metres on the WGS84 spheroid, so no projected CRS is
needed; `ST_Intersects` tests overlap with each 10 km buffer.

In [ ]:
proximity = sedona.sql("""
    SELECT
        m.mine_id,
        m.name,
        MIN(ST_DistanceSpheroid(m.geom, w.geometry)) / 1000.0        AS nearest_pa_km,
        COUNT(CASE WHEN ST_Intersects(m.buffer, w.geometry) THEN 1 END) AS pa_in_buffer
    FROM mines m
    CROSS JOIN wdpa w
    GROUP BY m.mine_id, m.name
""")
proximity.createOrReplaceTempView("proximity")

kba_overlap = sedona.sql("""
    SELECT
        m.mine_id,
        MAX(CASE WHEN ST_Intersects(m.buffer, k.geometry) THEN 1 ELSE 0 END) AS in_kba,
        MIN(ST_DistanceSpheroid(m.geom, k.geometry)) / 1000.0               AS nearest_kba_km
    FROM mines m
    CROSS JOIN kba k
    GROUP BY m.mine_id
""")
kba_overlap.createOrReplaceTempView("kba_overlap")

proximity.orderBy("nearest_pa_km").show(truncate=False)

**Raster × vector**, ecological value and recent deforestation *inside* each
buffer. `RS_ZonalStats` transforms each buffer to the raster's CRS on the fly, so
the 1 km BII grid and the 30 m Hansen grid both align to the same asset footprints
automatically. Forest loss is counted by binarizing Hansen's `loss_year`
(any year > 0 = loss) with `RS_MapAlgebra`, then summing.

In [ ]:
raster_metrics = sedona.sql("""
    SELECT
        m.mine_id,
        RS_ZonalStats(b.rast,  m.buffer, 1, 'mean', true) AS bii_mean,
        RS_ZonalStats(tc.rast, m.buffer, 1, 'mean', true) AS tree_cover_pct,
        RS_ZonalStats(
            RS_MapAlgebra(ly.rast, 'D', 'out = rast[0] > 0 ? 1.0 : 0.0;'),
            m.buffer, 1, 'sum', true
        ) AS forest_loss_pixels
    FROM mines m
    CROSS JOIN bii b
    CROSS JOIN tree_cover tc
    CROSS JOIN loss_year ly
""")
raster_metrics.createOrReplaceTempView("raster_metrics")
raster_metrics.show(truncate=False)

### Compose a per-asset nature-risk score

We join the four metric sets and combine them into a transparent, min-max
normalized composite. Weights are **illustrative**, the point is that all inputs
(two vector, two raster) arrived through one platform and one query language.
Higher score = more nature scrutiny warranted: closer to protected land, overlaps
a KBA, higher surrounding biodiversity value, and more recent forest loss.

In [ ]:
metrics = (
    mines_sdf.select("mine_id", "name")
    .join(proximity.select("mine_id", "nearest_pa_km", "pa_in_buffer"), "mine_id")
    .join(kba_overlap, "mine_id")
    .join(raster_metrics, "mine_id")
    .toPandas()
)

def norm(s, invert=False):
    rng = s.max() - s.min()
    if rng == 0:
        return pd.Series(0.0, index=s.index)
    z = (s - s.min()) / rng
    return 1 - z if invert else z

metrics["s_proximity"] = norm(metrics["nearest_pa_km"], invert=True)  # closer = higher
metrics["s_kba"]       = metrics["in_kba"].astype(float)
metrics["s_bii"]       = norm(metrics["bii_mean"].fillna(0))
metrics["s_loss"]      = norm(metrics["forest_loss_pixels"].fillna(0))

WEIGHTS = {"s_proximity": 0.30, "s_kba": 0.20, "s_bii": 0.25, "s_loss": 0.25}
metrics["nature_risk"] = sum(metrics[c] * w for c, w in WEIGHTS.items()).round(3)

ranked = metrics.sort_values("nature_risk", ascending=False)
ranked[[
    "name", "nearest_pa_km", "pa_in_buffer", "in_kba",
    "bii_mean", "forest_loss_pixels", "nature_risk",
]].reset_index(drop=True)

## 5. Visualize

In [ ]:
from sedona.spark.maps.SedonaKepler import SedonaKepler

# Attach the score back to the buffer geometries for mapping
scored_sdf = mines_sdf.join(
    sedona.createDataFrame(ranked[["mine_id", "nature_risk"]]),
    "mine_id",
)
display_df = scored_sdf.selectExpr("buffer AS geometry", "name", "nature_risk")

m = SedonaKepler.create_map(df=display_df, name="Asset nature risk")
SedonaKepler.add_df(m, df=wdpa_sdf.selectExpr("geometry", "name"), name="Protected areas")
m

## 6. Export (optional)

In [ ]:
# Save the scored portfolio for downstream reporting / BI
# scored_sdf.write.format("geoparquet").mode("overwrite").save(
#     os.getenv("USER_S3_PATH", "s3://your-bucket/") + "cecil/nature_risk_scores"
# )
print("Uncomment to export the scored portfolio.")

## From prompt to pipeline, the assistant layer

Everything above is what the **Wherobots Spatial Assistant** (and the
`cecil-for-wherobots` skill) generates from a plain-English request. The analyst
never hand-picks a dataset UUID or writes the credential boilerplate:

> **"Screen my mining portfolio for nature risk, proximity to protected areas
> and key biodiversity areas, plus biodiversity intactness and recent
> deforestation in each site's footprint."**

The assistant then:
1. **Discovers** the relevant Cecil datasets (WDPA, KBA, BII, Hansen) from the
   catalog and maps them to the question.
2. **Subscribes** to each for the portfolio's AOI through the Cecil SDK.
3. **Generates** this notebook, the correct `RS_*` and `ST_*` operators wired to
   the right bands and buffers.
4. **Runs** it on Wherobots compute and returns the ranked table + map.

That is the whole thesis: *ask in English, get raster + vector results on one
platform*, no juggling GDAL, rasterio, geopandas, PostGIS, CRS transforms, and
custom mosaic code.

## Scaling & honest caveats

- **Scale is a config change, not a rewrite.** This exact pipeline runs unchanged
  over 100k+ assets on Wherobots' distributed engine, swap the illustrative
  registry for an Overture Places / Buildings table (billions of features, already
  in the Wherobots Open Data Catalog) and raise the cluster size. The demo stays
  small deliberately so it runs cheaply end-to-end.
- **Dataset cost.** WDPA/KBA (IBAT) and BII (NHM) are commercial on Cecil; keep
  the AOI small, or use free substitutes (USGS PAD-US, Hansen) for a zero-cost run.
- **Illustrative coordinates and weights.** Asset locations and the composite
  weights are for demonstration; replace with your registry and risk model.
- **Extend to monitoring.** Re-running the Hansen block per `loss_year` value
  (`RS_MapAlgebra(... 'out = rast[0] == <year> ? 1.0 : 0.0;')`) turns this one-time
  screen into an annual change-detection loop, the natural follow-on for
  TNFD/CSRD re-disclosure and EUDR due diligence.
- **Verify bands after subscription.** Band order and data format (GeoTIFF vs
  Zarr) are only known once a subscription exists, the format check and band
  printout above surface both.